# Illustrating the core RAG pipeline
* question -> embed -> retrieve -> prompt -> LLM answer

The following depends on having an existing Chroma collection built beforehand (e.g. with our IndexBuilder notebook).

In the below, we:
1. `retrieve_chunks(question, k)`
    * Embeds the question
    * Asks Chroma for the top‑k similar chunks
    * Returns text + metadata + score
2. `build_rag_messages(question, chunks)`
    * Construct the prompt
      * System message = behavior instructions
      * User message = context block + question
      * Number the chunks ([1], [2], ...) so the model can cite them
3. Call the LLM (`llm_client.chat.completions.create`)
    * Send the prompt that now includes question + relevant context
    * Use a low temperature for grounded answering
4. Print the answer
    * `rag_answer` returns a plain string, so it’s easy to call

In [ ]:
from pathlib import Path
import json
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# This assumes that you have a "keys.py" file in this directory
# with NRP_TOK assigned the value of your NRP API token (required)
# and NRP_CACHE_SALT assigned your cache_salt value (optional)
import keys
NRP_TOK = keys.NRP_TOK
NRP_CACHE_SALT = keys.NRP_CACHE_SALT

# Global setup (run once)

Connect to the vector database:

In [ ]:
CHROMA_DIR = "./chromaNB3"
COLLECTION_NAME = "docs_v1"

print(f"Connecting to Chroma at {CHROMA_DIR}, with collection {COLLECTION_NAME}")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(COLLECTION_NAME)

Set up the embedding model:
* Assume we can read a manifest file to see which embedding model was used for the vector database

In [ ]:
manifest_fpath = Path(f"{CHROMA_DIR}/{COLLECTION_NAME}_manifest.json")
manifest = json.loads(manifest_fpath.read_text(encoding="utf-8"))

vector_db_embedding_model = manifest['config']['embedding_model']
print('Embedding model:', vector_db_embedding_model)

embed_model = SentenceTransformer(vector_db_embedding_model)

Set up the chat model:

In [ ]:
llm_client = OpenAI(api_key = NRP_TOK,
                    base_url = "https://ellm.nrp-nautilus.io/v1")

nrp_chat_model = 'gpt-oss'

# Retrieval

This embeds the question and retrieves top-k similar chunks from Chroma. [We assume we only have one string as the question, which we put into a list before being encoded.)

In [ ]:
def retrieve_chunks(question, k = 5):

    q_emb = embed_model.encode([question]).tolist()  # list of [vector]

    result = collection.query(
        query_embeddings=q_emb,
        n_results=k,
    )

    # Flatten results (Chroma returns lists-of-lists)
    docs = result["documents"][0]
    metas = result["metadatas"][0]
    dists = result["distances"][0]

    chunks = []
    for text, meta, dist in zip(docs, metas, dists):
        chunks.append({
            "text": text,
            "metadata": meta,
            "score": float(dist),
        })
    return chunks

Example output:

In [ ]:
question = "Who is Ben Winjum?"
retrieve_chunks(question, k = 15)

# Construct the Prompt

Here we format the retrieved chunks into a single string
* include citation number with source info
* separate chunks with "---"

In [ ]:
def make_context_block(chunks):
    """Format retrieved chunks into a context string with simple citations."""
    parts = []
    for i, ch in enumerate(chunks, start=1):
        src = ch["metadata"].get("source", "unknown")
        parts.append(
            f"[{i}] Source: {src}\n{ch['text']}"
        )
    return "\n\n---\n\n".join(parts)

The prompt constructed below includes both system content and user content.
* system message for instructions about behavior, including:
  * persona about being a helpful assistant
  * a guardrail that protects against hallucination if no relevant context is found.
  * instruction to cite sources
* user message includes:
  * context
  * question
  * further command to be precise and cite sources

In [ ]:
def build_rag_messages(question, chunks):

    context_block = make_context_block(chunks)

    system_msg = {
        "role": "system",
        "content": (
            "You are a helpful assistant that answers questions using the provided context.\n"
            "Use ONLY the information in the context to answer.\n"
            "If the answer is not in the context, say you don't know.\n"
            "When possible, cite sources like [1], [2]."
        ),
    }

    user_msg = {
        "role": "user",
        "content": f"""Context:
{context_block}

Question:
{question}

Answer (be concise and quote sources like [1], [2] when relevant):""",
    }

    return [system_msg, user_msg]

Testing the prompt construction:

In [ ]:
question = "Who is Ben Winjum?"
retrieve_chunks(question, k=2)

In [ ]:
question = "Who is Ben Winjum?"
chunks = retrieve_chunks(question, k=2)
prompt = build_rag_messages(question, chunks)
prompt

In [ ]:
print(prompt[1]['content'])

# RAG pipeline

* retrieve the relevant chunks
* construct the prompt
* call the LLM
* return the answer

In [ ]:
def rag_answer(question, k = 5):

    # 1) Retrieve relevant chunks
    chunks = retrieve_chunks(question, k=k)
    if not chunks:
        return "I couldn't find any relevant documents to answer that."

    # 2) Construct prompt
    messages = build_rag_messages(question, chunks)

    # 3) Call the LLM
    response = llm_client.chat.completions.create(
        model=nrp_chat_model,
        messages=messages,
        temperature=0.2,  # low temp for more determinism
        extra_body={"cache_salt": NRP_CACHE_SALT}
    )

    # 4) Get / return the answer
    answer = response.choices[0].message.content.strip()
    return answer

# Demo

In [ ]:
question = "What is my name?"

ans = rag_answer(question)

print("\nAssistant:", ans, "\n")

In [ ]:
question = "Who directed One Battle After Another?"

ans = rag_answer(question)

print("\nAssistant:", ans, "\n")

In [ ]:
question = "Who is Ben Winjum?"

ans = rag_answer(question)

print("\nAssistant:", ans, "\n")

In [ ]:
question = "Who teaches the Large Language Models class?"

ans = rag_answer(question)

print("\nAssistant:", ans, "\n")